# Copilot Licensed Users Loader

Ingests one or more **Copilot licensed user list** CSVs (typically exported from MAC) into a flat Delta table consumed by the **AI-in-One Dashboard** and **AI Business Value Dashboard** Fabric variants.

**Inputs**: CSV(s) landed in `Files/licensed_raw/` of this Lakehouse. Accepted UPN column names: `User Principal Name`, `userPrincipalName`, `UserPrincipalName`, `User principal name`. Accepted licence column names: `Has license`, `Has Licence`, `HasLicense`, `HasCopilot`, `Has Copilot License`, `Has Copilot license assigned`, `isUser`, etc.

**Output**: Lakehouse Delta table `dbo.copilot_licensed_users`. Delta tables disallow spaces / `,;{}()\n\t=` in column names, so the loader sanitises all column names to underscore-safe form (`User Principal Name` → `User_Principal_Name`, `Has license` → `Has_license`, `Report Refresh Date` → `Report_Refresh_Date`, etc.). The PBIT's `sourceColumn` references are aligned with these names.

**Schedule**: run on the same cadence as your licence export (typically weekly/monthly).


## 1. Configuration

In [ ]:
RAW_PATH     = 'Files/licensed_raw/*.csv'         # raw licence-export CSVs
OUTPUT_TABLE = 'copilot_licensed_users'           # Delta table consumed by the PBIT
WRITE_MODE   = 'overwrite'                        # 'overwrite' for full snapshots; switch to 'append' if appending

## 2. Read raw CSVs and detect column variants
Mirrors the M-query renaming logic so users dropping different MAC export shapes still produce the same Delta schema.

In [ ]:
import re
from pyspark.sql import functions as F

raw = (spark.read
    .option('header', 'true')
    .option('multiline', 'true')
    .option('escape', '"')
    .csv(RAW_PATH))

print(f'Raw rows: {raw.count():,}')
print('Detected columns:', raw.columns)

In [ ]:
# Column-variant detection — find the actual UPN and Has-license columns
upn_variants = ['User Principal Name', 'userPrincipalName', 'UserPrincipalName', 'User principal name']
license_variants = [
    'Has license', 'Has License', 'Has Licence', 'Has licence',
    'HasLicense', 'HasCopilot', 'Has Copilot', 'Has Copilot License',
    'Has Copilot license', 'HasCopilotLicense',
    'Has Copilot License Assigned', 'Has Copilot license assigned',
    'isUser'
]

actual_upn     = next((c for c in upn_variants     if c in raw.columns), None)
actual_license = next((c for c in license_variants if c in raw.columns), None)

print(f'UPN column detected:      {actual_upn}')
print(f'Licence column detected:  {actual_license}')

df = raw

# Rename UPN variant to canonical 'User Principal Name' (then we sanitize below)
if actual_upn is None:
    df = df.withColumn('User Principal Name', F.lit(None).cast('string'))
elif actual_upn != 'User Principal Name':
    df = df.withColumnRenamed(actual_upn, 'User Principal Name')

# Rename licence variant to canonical 'Has license'
if actual_license is None:
    df = df.withColumn('Has license', F.lit('Unknown'))
elif actual_license != 'Has license':
    df = df.withColumnRenamed(actual_license, 'Has license')

## 3. Add normalized join key + dedupe

In [ ]:
df = df.withColumn(
    'UPN_Normalized',
    F.when(F.col('`User Principal Name`').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('`User Principal Name`').cast('string'))))
)

# Dedupe on the normalized key — keep first row per UPN_Normalized
df = df.dropDuplicates(['UPN_Normalized'])

print(f'After dedupe: {df.count():,} rows')

## 4. Sanitize column names for Delta
Delta rejects column names containing ` ,;{}()\n\t=`. Replace each invalid character with `_` (e.g. `User Principal Name` → `User_Principal_Name`).

In [ ]:
_INVALID = re.compile(r'[ ,;{}()\n\t=]')

def sanitize(name: str) -> str:
    return _INVALID.sub('_', name)

df = df.toDF(*[sanitize(c) for c in df.columns])
print('Sanitized columns:', df.columns)

## 5. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

print(f'Rows written to {OUTPUT_TABLE}: {spark.table(OUTPUT_TABLE).count():,}')

## 6. Verify
Spot-check the output. Expect every row to have `UPN_Normalized` populated and a non-null `Has_license`.

In [ ]:
spark.table(OUTPUT_TABLE).select(
    'User_Principal_Name', 'Has_license', 'UPN_Normalized'
).show(10, truncate=False)

spark.table(OUTPUT_TABLE).groupBy('Has_license').count().orderBy(F.desc('count')).show(20, truncate=False)

---
**Connect the PBIT**: this table is consumed by the `Copilot Licensed` query in the AI-in-One and AI Business Value dashboards. Once this notebook has run, you can leave the `Copilot Licensed Users` parameter blank when opening the PBIT — refresh will source from `dbo.copilot_licensed_users` directly via the Fabric SQL endpoint.